# 06 优化器：SGD、Momentum、Adam

上一节我们讲了训练流程：前向传播、计算损失、反向传播、更新参数。

这一节专门讲“更新参数”这一步。

优化器的作用可以先用一句话理解：**优化器决定模型参数每一步怎么走。**

## 1. 为什么需要优化器

反向传播会算出梯度：

$$
\nabla_{\theta}\mathcal{L}
$$

梯度告诉我们：如果参数沿着这个方向变化，损失会上升得最快。

既然我们希望损失下降，就沿着反方向走：

$$
\theta \leftarrow \theta - \eta \nabla_{\theta}\mathcal{L}
$$

这就是最基础的梯度下降。

但真实训练中，问题没有这么理想：

- 梯度可能很抖。
- 有些方向走得太慢。
- 有些方向步子太大。
- 不同参数适合不同的更新幅度。

优化器就是为了解决“怎么更聪明地使用梯度”这个问题。

## 2. 先区分两个概念：梯度和优化器

梯度是方向信息。

优化器是走路策略。

可以这样理解：

```text
梯度：告诉你哪里更陡
优化器：决定你往哪里走、走多大步、要不要参考过去的方向
```

最简单的优化器只看当前梯度。

更复杂的优化器会参考历史梯度，甚至会给每个参数单独调整步幅。

## 3. Batch Gradient Descent 是什么

最朴素的梯度下降，是每次用全部训练集计算一次梯度，再更新一次参数。

如果训练集是：

$$
\mathcal{D}=\{(x^{(1)},y^{(1)}),\dots,(x^{(m)},y^{(m)})\}
$$

整体损失是：

$$
\mathcal{L}=\frac{1}{m}\sum_{i=1}^{m}\mathcal{L}^{(i)}
$$

然后更新：

$$
\theta \leftarrow \theta-\eta\nabla_{\theta}\mathcal{L}
$$

这种方法方向比较稳定，因为它看完了所有样本才决定怎么走。

但缺点也明显：如果数据量很大，每一步都要算完整训练集，太慢。

## 4. SGD 是什么

SGD 的全称是 Stochastic Gradient Descent，随机梯度下降。

严格来说，最原始的 SGD 是每次随机抽一个样本来更新参数。

如果抽到第 $i$ 个样本，更新就是：

$$
\theta \leftarrow \theta-\eta\nabla_{\theta}\mathcal{L}^{(i)}
$$

实际深度学习中，我们通常用一个 mini-batch 来计算梯度，这也常被习惯性叫作 SGD。

如果一个 batch 里有 $B$ 个样本：

$$
\mathcal{L}_{batch}=\frac{1}{B}\sum_{i=1}^{B}\mathcal{L}^{(i)}
$$

更新为：

$$
\theta \leftarrow \theta-\eta
\nabla_{\theta}\mathcal{L}_{batch}
$$

SGD 的核心是：不用等看完整个训练集，先看一小批数据，就更新一步。

## 5. SGD 为什么会抖

SGD 每次只看一小批样本。这个 batch 不能完全代表整个训练集。

所以这次 batch 可能告诉你往左走，下次 batch 又告诉你稍微往右走。

数学上可以理解为：batch 梯度只是整体梯度的一个近似：

$$
\nabla_{\theta}\mathcal{L}_{batch}
\approx
\nabla_{\theta}\mathcal{L}_{all}
$$

因为只是近似，所以会有噪声。

SGD 的优点是更新频繁、计算压力小、有噪声时有时还能跳出一些不好的位置。

SGD 的缺点是路线不够平滑，可能在正确方向附近来回摆动。

## 6. Momentum 为什么出现

Momentum 的意思是动量。

它想解决 SGD 的一个问题：每一步只看当前梯度，容易被当前 batch 的噪声带偏。

Momentum 的想法是：不要只看当前这一步，也要参考过去几步的大方向。

可以这样理解：

```text
SGD：每一步都重新看当前梯度，然后立刻改方向
Momentum：记住之前的方向，当前梯度只是逐渐修正方向
```

这就像一个球从山坡往下滚。它不会因为地面一点小颠簸就马上改变方向，因为它有惯性。

## 7. Momentum 的公式怎么读

Momentum 会维护一个速度变量 $v_t$。

先更新速度：

$$
v_t=\beta v_{t-1}+(1-\beta)\nabla_{\theta}\mathcal{L}_t
$$

再用速度更新参数：

$$
\theta_t=\theta_{t-1}-\eta v_t
$$

这里：

- $v_t$ 是当前累积出来的更新方向。
- $v_{t-1}$ 是过去的方向。
- $\nabla_{\theta}\mathcal{L}_t$ 是当前梯度。
- $\beta$ 控制多大程度保留过去方向。

如果 $\beta$ 比较大，比如 $0.9$，意思是：

$$
v_t=0.9v_{t-1}+0.1\nabla_{\theta}\mathcal{L}_t
$$

当前方向主要参考过去，同时慢慢吸收新梯度。

## 8. Momentum 解决了什么

Momentum 的作用主要有两个。

第一，减少来回摆动。

如果某个方向的梯度一会儿正、一会儿负，Momentum 会把这些相反方向互相抵消，让路线更平滑。

第二，加速稳定方向上的前进。

如果某个方向的梯度长期都差不多，Momentum 会不断积累这个方向的速度，让模型在这个方向走得更快。

所以 Momentum 不是改变优化目标，而是让参数更新更稳、更有惯性。

## 9. Adam 为什么出现

Momentum 解决了“方向太抖”的问题，但还有一个问题：不同参数可能需要不同的步幅。

有些参数梯度经常很大，如果按同一个学习率更新，可能步子太猛。

有些参数梯度经常很小，如果按同一个学习率更新，可能几乎走不动。

Adam 的想法是：

```text
既参考历史方向，又根据每个参数的梯度大小自动调整步幅
```

所以 Adam 可以理解成：Momentum 思想 + 自适应学习率思想。

## 10. Adam 里的两个记忆

Adam 会同时维护两个量。

第一个是梯度的一阶矩估计，可以先理解成梯度的移动平均：

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

其中：

$$
g_t=\nabla_{\theta}\mathcal{L}_t
$$

$m_t$ 记录的是最近一段时间梯度的大方向。

第二个是梯度平方的移动平均：

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
$$

$v_t$ 记录的是最近一段时间梯度的大小。

所以 Adam 同时问两个问题：

```text
方向大概往哪边？
这个参数最近梯度大不大？
```

## 11. Adam 的参数更新怎么理解

Adam 的更新可以简化理解为：

$$
\theta_t=\theta_{t-1}-\eta\frac{m_t}{\sqrt{v_t}+\epsilon}
$$

先不用纠结偏差修正细节，先看这个式子的含义。

分子 $m_t$ 表示方向。

分母 $\sqrt{v_t}+\epsilon$ 表示根据梯度大小调整步幅。

如果某个参数最近梯度一直很大，那么 $v_t$ 会变大，分母变大，实际更新步幅会被压小。

如果某个参数最近梯度一直很小，那么 $v_t$ 较小，分母较小，实际更新不会被过度压缩。

$\epsilon$ 是一个很小的数，用来防止分母为 $0$。

## 12. Adam 为什么常被初学者优先使用

Adam 通常比较好用，是因为它对学习率没那么敏感。

SGD 很依赖学习率。如果学习率没调好，可能训练很慢，或者震荡严重。

Adam 会自动根据每个参数的梯度情况调整实际步幅，所以初始训练时常常更稳。

但这不代表 Adam 永远最好。

在一些任务中，SGD 加 Momentum 可能最终泛化更好。Adam 的优势是上手快、收敛快、调参压力小。

## 13. SGD、Momentum、Adam 对比

| 优化器 | 核心想法 | 优点 | 缺点 |
|---|---|---|---|
| SGD | 直接沿负梯度方向走 | 简单，泛化常不错 | 路线抖，学习率敏感 |
| Momentum | 参考历史方向，加入惯性 | 减少震荡，加速稳定方向 | 多一个动量参数 |
| Adam | 历史方向 + 自适应步幅 | 收敛快，调参较容易 | 有时泛化不一定最好 |

从学习顺序上，可以这样理解：

```text
SGD：知道当前坡往哪里下
Momentum：不只看当前坡，还带着过去的惯性
Adam：既带惯性，又给不同参数调整不同步幅
```

## 14. 学习率和优化器的关系

优化器再聪明，也离不开学习率 $\eta$。

学习率控制基础步幅：

$$
\theta \leftarrow \theta-\eta \cdot \text{update direction}
$$

SGD 中，update direction 基本就是梯度。

Momentum 中，update direction 是累积速度。

Adam 中，update direction 是经过一阶矩和二阶矩调整后的方向。

但无论哪种优化器，学习率太大都可能震荡或发散；学习率太小都可能训练很慢。

## 15. 初学阶段怎么选优化器

入门阶段可以先按这个顺序理解和使用：

1. 先理解 SGD，因为它是所有优化器的基础。
2. 再理解 Momentum，因为它解释了为什么历史梯度有用。
3. 最后理解 Adam，因为它是很多深度学习任务的常用起点。

实践时，如果只是想先把模型训练起来，Adam 往往是一个友好的起点。

如果想研究模型最终泛化表现，可以再尝试 SGD + Momentum。

## 16. 本节总结

这一节的逻辑链是：

```text
反向传播算出梯度
-> 优化器决定怎么使用梯度
-> SGD 直接用当前梯度
-> Momentum 加入历史方向，减少震荡
-> Adam 同时考虑历史方向和梯度大小，自适应调整步幅
```

先记住三个核心公式：

SGD：

$$
\theta \leftarrow \theta-\eta\nabla_{\theta}\mathcal{L}
$$

Momentum：

$$
v_t=\beta v_{t-1}+(1-\beta)\nabla_{\theta}\mathcal{L}_t
$$

Adam 简化理解：

$$
\theta_t=\theta_{t-1}-\eta\frac{m_t}{\sqrt{v_t}+\epsilon}
$$

下一节可以继续讲正则化：为什么模型会过拟合，以及 Dropout、权重衰减、早停分别在解决什么问题。